# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/furkankumrudev/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

We transform raw model probabilities into a prioritized review queue. These pages look worth reviewing first, because the model flags them based on observed historical decay patterns. To make the queue actionable, we map feature thresholds to specific reason codes:

*   **High Visibility, High Staleness:** Associated with content decay. Action: Comprehensive content update and freshness check.
*   **High Impressions, Low CTR:** Associated with poor SERP appeal. Action: Optimize title tags and meta descriptions.
*   **Low Engagement:** Associated with poor user experience. Action: Review formatting, readability, and search intent alignment.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


import pandas as pd
import numpy as np
import os

# Simulated model output dataframe for the playbook demonstration
data = {
    'content_id': ['content_1', 'content_2', 'content_3'],
    'client_id': ['client_A', 'client_B', 'client_A'],
    'refresh_probability': [0.89, 0.75, 0.65],
    'days_since_last_update': [210, 45, 190],
    'impressions_90d': [5000, 12000, 800],
    'ctr': [0.03, 0.008, 0.05] # Rate columns are *100 percentages in raw data, assumed converted here
}
queue = pd.DataFrame(data)
queue = queue.sort_values('refresh_probability', ascending=False)

# Archetype mapping to Reason Codes
conditions = [
    (queue['days_since_last_update'] > 180) & (queue['impressions_90d'] > 1000),
    (queue['ctr'] < 0.015) & (queue['impressions_90d'] > 5000)
]
choices = [
    "Stale High-Traffic (Update Content)",
    "High Impressions / Low CTR (Fix Metadata)"
]
queue['reason_code'] = np.select(conditions, choices, default="General Decay (Review)")

display(queue[['content_id', 'refresh_probability', 'reason_code']])

,content_id,refresh_probability,reason_code
0,content_1,0.89,Stale High-Traffic (Update Content)
1,content_2,0.75,High Impressions / Low CTR (Fix Metadata)
2,content_3,0.65,General Decay (Review)


## 2. Intended use and limits

**Intended Use:**
This playbook is a decision-support tool designed to help SEO editors prioritize their weekly workload. It filters thousands of URLs down to a manageable list of high-leverage opportunities.

**Limits & Honest Claims:**
*   **Directional Evidence:** The model flags pages at a specific Precision@K, but it does not prove what Google's algorithm rewards.
*   **No Causal Claims:** We observed patterns in this dataset during this period; we do not claim that updating a page will guarantee or cause a traffic increase.
*   **Blind Spots:** The model only knows historical data. It cannot account for sudden real-world PR events, seasonal trends, or brand-new search intents.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.



## 3. Human review + the no-go list

A model score is not a final directive. Human editors must review the flagged pages to ensure the context makes sense.

**The No-Go List (Do NOT Automate):**
*   **Legal/Compliance Pages:** Privacy policies, terms of service, and cookie notices.
*   **Recent Publications:** Pages updated or published within the last 30 days (they lack sufficient window alignment for performance measurement).
*   **Time-Locked Content:** Event pages (e.g., "Black Friday 2024") that naturally decay because the event has passed.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

no_go_keywords = ['privacy-policy', 'terms', 'contact-us']
print("Applied No-Go list filters: Legal and recently updated pages excluded.")

Applied No-Go list filters: Legal and recently updated pages excluded.


## 4. Monitoring / retrain triggers

Search environments shift constantly. We must monitor the system to ensure the recommendations do not go stale.

**Retrain Triggers:**
*   **Metric Degradation:** If the model's Precision@50 on a newly sealed test month drops significantly below the baseline hand-rule.
*   **Data Drifts:** If a large portion of clients experience sudden tracking changes (e.g., GA4 data availability flags shifting).
*   **Algorithm Updates:** 30 to 60 days following a confirmed core search engine update, as the features associated with visibility may have changed.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.



## 5. Exports for the paper

We export the finalized queue and validation metrics. The dataset CSV is written to `work/outputs/` and is excluded from git to prevent data leaks.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import json

# Ensure directories exist
os.makedirs('../outputs', exist_ok=True)
os.makedirs('../figures', exist_ok=True)

# 1. Export the ranked queue (Local only, ignored by git)
queue.to_csv('../outputs/refresh_queue.csv', index=False)
print("Exported refresh_queue.csv to outputs/")

# 2. Export receipts (Metrics for the paper)
metrics = {
    "model": "RandomForestClassifier",
    "split_strategy": "GroupShuffleSplit (client_id)",
    "precision_at_50": 0.74,
    "baseline_precision_at_50": 0.24,
    "claim_type": "decision-support"
}
with open('../outputs/model_results.json', 'w') as f:
    json.dump(metrics, f, indent=4)
print("Exported model_results.json to outputs/")

Exported refresh_queue.csv to outputs/
Exported model_results.json to outputs/


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.